# MLP on MNIST — Real Dataset Experiments

**Goal:** Train a multi-layer perceptron on MNIST (handwritten digits) using PyTorch.
Run optimizer and regularization experiments, visualize learned weights, and analyze
failure modes.

**Target:** >97% test accuracy on MNIST.

**What you'll do:**
1. Load and explore MNIST
2. Build a configurable MLP in PyTorch (batch norm, dropout)
3. Clean training loop with validation tracking
4. Compare SGD vs Adam convergence curves
5. Compare dropout rates (none, 0.3, 0.5) — train vs val curves
6. Visualize learned first-layer weights as 28x28 images
7. Confusion matrix and misclassified examples

**Prerequisites:** Complete `01_from_scratch.ipynb` first.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device:          {DEVICE}')

## Part 1: Load and Explore MNIST

MNIST contains 70,000 grayscale images of handwritten digits 0–9, each 28×28 pixels.
- Training set: 60,000 images
- Test set: 10,000 images

We normalize each pixel to zero mean and unit standard deviation using the dataset-wide
statistics: mean=0.1307, std=0.3081.

In [ ]:
# Dataset constants
MNIST_MEAN = 0.1307
MNIST_STD  = 0.3081
N_CLASSES  = 10
IMAGE_SIZE = 28 * 28    # flattened input dimension
BATCH_SIZE = 128
DATA_ROOT  = './data'

# Transform: convert PIL image → tensor, then normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,))
])

# Download and load
train_dataset = datasets.MNIST(DATA_ROOT, train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(DATA_ROOT, train=False, download=True, transform=transform)

# Split training set into train and validation
VAL_SIZE = 5000
train_subset, val_subset = torch.utils.data.random_split(
    train_dataset, [len(train_dataset) - VAL_SIZE, VAL_SIZE],
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_subset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Training samples:    {len(train_subset):,}')
print(f'Validation samples:  {len(val_subset):,}')
print(f'Test samples:        {len(test_dataset):,}')
print(f'Batch size:          {BATCH_SIZE}')
print(f'Batches per epoch:   {len(train_loader)}')

In [ ]:
# Visualize sample images from each class
N_SAMPLES_PER_CLASS = 5
CLASS_NAMES = [str(d) for d in range(10)]

fig, axes = plt.subplots(N_CLASSES, N_SAMPLES_PER_CLASS, figsize=(N_SAMPLES_PER_CLASS * 2, N_CLASSES * 2))

# Collect one sample per class
class_samples = {i: [] for i in range(N_CLASSES)}
for img, label in train_dataset:
    if len(class_samples[label.item()]) < N_SAMPLES_PER_CLASS:
        class_samples[label.item()].append(img)
    if all(len(v) >= N_SAMPLES_PER_CLASS for v in class_samples.values()):
        break

for row, cls in enumerate(range(N_CLASSES)):
    for col, img in enumerate(class_samples[cls]):
        ax = axes[row, col]
        ax.imshow(img.squeeze().numpy(), cmap='gray')
        ax.axis('off')
        if col == 0:
            ax.set_title(f'Digit {cls}', fontsize=10, loc='left')

plt.suptitle('MNIST Sample Images (5 per class)', fontsize=14)
plt.tight_layout()
plt.show()

# Print data statistics
sample_img, sample_label = train_dataset[0]
print(f'Image shape: {sample_img.shape}  (C, H, W)')
print(f'Pixel min:   {sample_img.min():.3f}, max: {sample_img.max():.3f}')
print(f'Class labels: {list(range(N_CLASSES))}')

## Part 2: Build Configurable MLP in PyTorch

We build a clean, configurable MLP with:
- 3 hidden layers (configurable sizes)
- Batch normalization after each linear layer (before activation)
- Dropout after each activation
- Linear output layer (cross-entropy loss handles softmax)

The `dropout_rate=0.0` setting disables dropout entirely, so we can ablate it later.

In [ ]:
class MNISTClassifier(nn.Module):
    """
    Configurable 3-hidden-layer MLP for MNIST classification.

    Architecture per hidden layer:
        Linear → BatchNorm1d → ReLU → Dropout
    Output:
        Linear (logits, N_CLASSES)
    """

    def __init__(
        self,
        input_dim=IMAGE_SIZE,
        hidden_dims=(512, 256, 128),
        n_classes=N_CLASSES,
        dropout_rate=0.3,
        use_batchnorm=True,
    ):
        """
        Args:
            input_dim:    number of input features (28*28 = 784)
            hidden_dims:  tuple of hidden layer sizes
            n_classes:    number of output classes
            dropout_rate: dropout probability (0.0 disables dropout)
            use_batchnorm: whether to apply batch normalization
        """
        super().__init__()

        self.flatten = nn.Flatten()

        hidden_layers = []
        prev_dim = input_dim

        for h_dim in hidden_dims:
            hidden_layers.append(nn.Linear(prev_dim, h_dim))
            if use_batchnorm:
                hidden_layers.append(nn.BatchNorm1d(h_dim))
            hidden_layers.append(nn.ReLU())
            if dropout_rate > 0.0:
                hidden_layers.append(nn.Dropout(dropout_rate))
            prev_dim = h_dim

        self.hidden = nn.Sequential(*hidden_layers)
        self.output = nn.Linear(prev_dim, n_classes)

    def forward(self, x):
        """
        Args:
            x: (N, 1, 28, 28) image batch
        Returns:
            logits: (N, n_classes)
        """
        x = self.flatten(x)      # (N, 784)
        x = self.hidden(x)       # (N, last_hidden_dim)
        return self.output(x)    # (N, n_classes)


# Inspect model
model_demo = MNISTClassifier(dropout_rate=0.3)
model_demo = model_demo.to(DEVICE)

n_params = sum(p.numel() for p in model_demo.parameters() if p.requires_grad)
print(f'Model architecture:')
print(model_demo)
print(f'\nTotal trainable parameters: {n_params:,}')

# Verify output shape
dummy_batch = torch.zeros(4, 1, 28, 28).to(DEVICE)
logits_demo = model_demo(dummy_batch)
print(f'\nInput shape:  {dummy_batch.shape}')
print(f'Output shape: {logits_demo.shape}  (expected (4, 10))')

## Part 3: Training Loop

A clean, reusable training loop that:
- Trains one epoch and returns mean loss + accuracy
- Evaluates on validation set (model.eval(), no gradient)
- Tracks train loss, val loss, train accuracy, and val accuracy

The `train_one_epoch` / `evaluate` pattern is standard PyTorch practice.

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """
    Train model for one epoch.

    Args:
        model:     nn.Module
        loader:    DataLoader (train)
        optimizer: torch optimizer
        criterion: loss function
        device:    torch.device
    Returns:
        mean_loss: float
        accuracy:  float in [0, 1]
    """
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """
    Evaluate model on a data loader.

    Args:
        model:     nn.Module
        loader:    DataLoader (val or test)
        criterion: loss function
        device:    torch.device
    Returns:
        mean_loss: float
        accuracy:  float in [0, 1]
    """
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)

        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


def train_model(model, train_loader, val_loader, optimizer, n_epochs, device, verbose=True):
    """
    Full training loop with per-epoch validation.

    Args:
        model:        nn.Module
        train_loader: DataLoader
        val_loader:   DataLoader
        optimizer:    torch optimizer
        n_epochs:     int
        device:       torch.device
        verbose:      print progress every 5 epochs
    Returns:
        history: dict with lists 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(1, n_epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc     = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if verbose and (epoch % 5 == 0 or epoch == 1):
            print(f'Epoch {epoch:3d}/{n_epochs}  '
                  f'train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  '
                  f'train_acc={train_acc:.3f}  val_acc={val_acc:.3f}')

    return history


print('Training utilities defined.')

## Part 4: Optimizer Comparison — SGD vs Adam

We train two identical models from scratch — one with SGD and one with Adam — and
compare their loss and accuracy convergence curves side by side.

Adam typically converges faster in the early epochs. SGD with a well-tuned learning
rate can sometimes reach better final accuracy, but requires more hyperparameter tuning.

In [ ]:
N_EPOCHS_OPTIM = 20
HIDDEN_DIMS    = (512, 256, 128)
DROPOUT_RATE   = 0.3

# SGD model
torch.manual_seed(SEED)
model_sgd = MNISTClassifier(hidden_dims=HIDDEN_DIMS, dropout_rate=DROPOUT_RATE).to(DEVICE)
optimizer_sgd = torch.optim.SGD(model_sgd.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)

print('Training with SGD (lr=0.1, momentum=0.9)...')
history_sgd = train_model(
    model_sgd, train_loader, val_loader, optimizer_sgd, N_EPOCHS_OPTIM, DEVICE
)

In [ ]:
# Adam model
torch.manual_seed(SEED)
model_adam = MNISTClassifier(hidden_dims=HIDDEN_DIMS, dropout_rate=DROPOUT_RATE).to(DEVICE)
optimizer_adam = torch.optim.Adam(model_adam.parameters(), lr=1e-3, weight_decay=1e-4)

print('Training with Adam (lr=1e-3)...')
history_adam = train_model(
    model_adam, train_loader, val_loader, optimizer_adam, N_EPOCHS_OPTIM, DEVICE
)

In [ ]:
# Plot convergence comparison
epochs_range = range(1, N_EPOCHS_OPTIM + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(epochs_range, history_sgd['val_loss'],  label='SGD val loss',  color='steelblue', linewidth=2)
axes[0].plot(epochs_range, history_adam['val_loss'], label='Adam val loss', color='coral',     linewidth=2)
axes[0].plot(epochs_range, history_sgd['train_loss'],  linestyle='--', color='steelblue', alpha=0.5, label='SGD train loss')
axes[0].plot(epochs_range, history_adam['train_loss'], linestyle='--', color='coral',     alpha=0.5, label='Adam train loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Loss: SGD vs Adam')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(epochs_range, history_sgd['val_acc'],  label='SGD val acc',  color='steelblue', linewidth=2)
axes[1].plot(epochs_range, history_adam['val_acc'], label='Adam val acc', color='coral',     linewidth=2)
axes[1].plot(epochs_range, history_sgd['train_acc'],  linestyle='--', color='steelblue', alpha=0.5, label='SGD train acc')
axes[1].plot(epochs_range, history_adam['train_acc'], linestyle='--', color='coral',     alpha=0.5, label='Adam train acc')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy: SGD vs Adam')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.8, 1.0)

plt.tight_layout()
plt.show()

print(f'Final val accuracy — SGD: {history_sgd["val_acc"][-1]:.3f},  Adam: {history_adam["val_acc"][-1]:.3f}')
print('Observe: Adam usually reaches high accuracy faster in the early epochs.')

## Part 5: Regularization Experiments — Dropout Comparison

We compare three dropout settings on identical models:
- **No dropout** (dropout=0.0): maximum capacity, highest overfitting risk
- **Dropout=0.3**: moderate regularization
- **Dropout=0.5**: strong regularization

We plot **train vs val** curves for each — the gap between them shows the overfitting.
On MNIST the dataset is large enough that overfitting is mild; on smaller datasets the
difference would be more pronounced.

In [ ]:
N_EPOCHS_REG = 25
DROPOUT_RATES = [0.0, 0.3, 0.5]
COLORS = ['steelblue', 'coral', 'forestgreen']

histories_reg = {}

for dropout_rate in DROPOUT_RATES:
    torch.manual_seed(SEED)
    model_reg = MNISTClassifier(
        hidden_dims=HIDDEN_DIMS,
        dropout_rate=dropout_rate,
        use_batchnorm=True
    ).to(DEVICE)
    opt_reg = torch.optim.Adam(model_reg.parameters(), lr=1e-3, weight_decay=1e-4)

    print(f'Training dropout={dropout_rate}...')
    histories_reg[dropout_rate] = train_model(
        model_reg, train_loader, val_loader, opt_reg, N_EPOCHS_REG, DEVICE, verbose=False
    )
    final_val_acc = histories_reg[dropout_rate]['val_acc'][-1]
    print(f'  Final val accuracy: {final_val_acc:.3f}')

In [ ]:
# Plot train vs val for each dropout setting
epochs_reg = range(1, N_EPOCHS_REG + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (dp, color) in zip(axes, zip(DROPOUT_RATES, COLORS)):
    hist = histories_reg[dp]
    ax.plot(epochs_reg, hist['train_loss'], label='Train loss', color=color,     linewidth=2)
    ax.plot(epochs_reg, hist['val_loss'],   label='Val loss',   color=color,     linewidth=2, linestyle='--')

    # Shade the generalization gap
    ax.fill_between(
        epochs_reg,
        hist['train_loss'],
        hist['val_loss'],
        alpha=0.15, color=color, label='Gap'
    )

    label = 'No dropout' if dp == 0.0 else f'Dropout={dp}'
    final_gap = hist['val_loss'][-1] - hist['train_loss'][-1]
    ax.set_title(f'{label}\n(final gap={final_gap:.4f})', fontsize=11)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Regularization Comparison: Train vs Val Loss by Dropout Rate', fontsize=13)
plt.tight_layout()
plt.show()

print('Observe: higher dropout → smaller generalization gap (less overfitting).')
print('But too much dropout may slow convergence or hurt final accuracy.')

## Part 6: Train Final Model to Target Accuracy

We now train a full model with our best settings for enough epochs to reach >97% accuracy.
This uses a cosine annealing learning rate schedule.

In [ ]:
N_EPOCHS_FINAL = 30

torch.manual_seed(SEED)
model_final = MNISTClassifier(
    hidden_dims=(512, 256, 128),
    dropout_rate=0.3,
    use_batchnorm=True
).to(DEVICE)

optimizer_final = torch.optim.Adam(
    model_final.parameters(), lr=1e-3, weight_decay=1e-4
)

# Cosine annealing schedule: lr decays from lr_max to lr_min over N_EPOCHS_FINAL steps
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_final, T_max=N_EPOCHS_FINAL, eta_min=1e-5
)

criterion_final = nn.CrossEntropyLoss()
history_final = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

print(f'Training final model for {N_EPOCHS_FINAL} epochs...')
for epoch in range(1, N_EPOCHS_FINAL + 1):
    train_loss, train_acc = train_one_epoch(
        model_final, train_loader, optimizer_final, criterion_final, DEVICE
    )
    val_loss, val_acc = evaluate(model_final, val_loader, criterion_final, DEVICE)
    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()

    history_final['train_loss'].append(train_loss)
    history_final['val_loss'].append(val_loss)
    history_final['train_acc'].append(train_acc)
    history_final['val_acc'].append(val_acc)
    history_final['lr'].append(current_lr)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}  '
              f'train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  '
              f'val_acc={val_acc:.4f}  lr={current_lr:.2e}')

# Final test accuracy
test_loss_final, test_acc_final = evaluate(model_final, test_loader, criterion_final, DEVICE)
print(f'\nFinal test accuracy: {test_acc_final:.4f}  (target: >0.97)')
print(f'Final test loss:     {test_loss_final:.4f}')

In [ ]:
# Plot final model training curves including LR schedule
epochs_final = range(1, N_EPOCHS_FINAL + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs_final, history_final['train_loss'], label='Train', color='steelblue', linewidth=2)
axes[0].plot(epochs_final, history_final['val_loss'],   label='Val',   color='coral',     linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curve (Final Model)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_final, history_final['train_acc'], label='Train', color='steelblue', linewidth=2)
axes[1].plot(epochs_final, history_final['val_acc'],   label='Val',   color='coral',     linewidth=2)
axes[1].axhline(0.97, color='gray', linestyle=':', label='Target 97%')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curve (Final Model)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0.9, 1.0)

# LR schedule
axes[2].plot(epochs_final, history_final['lr'], color='purple', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Cosine Annealing LR Schedule')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Final Model Training (Test Acc = {test_acc_final:.4f})', fontsize=13)
plt.tight_layout()
plt.show()

## Part 7: Weight Visualization

The first layer's weights have shape `(n_hidden, 784)`. Each row is a weight vector that
can be reshaped to 28×28 and visualized as an image.

These learned templates show what patterns each neuron responds to. We expect to see
stroke patterns, curves, and edges that correspond to parts of digit shapes.

In [ ]:
# Extract the first linear layer's weights
# The first module in model_final.hidden is the first Linear layer
first_linear = None
for module in model_final.hidden:
    if isinstance(module, nn.Linear):
        first_linear = module
        break

W_first = first_linear.weight.detach().cpu().numpy()  # (n_hidden, 784)
print(f'First layer weight shape: {W_first.shape}')

# Display 64 learned filters as 28x28 images
N_SHOW = 64
GRID_COLS = 8
GRID_ROWS = N_SHOW // GRID_COLS

fig, axes = plt.subplots(GRID_ROWS, GRID_COLS, figsize=(GRID_COLS * 1.5, GRID_ROWS * 1.5))

for i, ax in enumerate(axes.ravel()):
    if i >= N_SHOW:
        ax.set_visible(False)
        continue
    weight_img = W_first[i].reshape(28, 28)
    # Normalize each filter to [0,1] for display
    vmax = np.abs(weight_img).max()
    ax.imshow(weight_img, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.axis('off')

plt.suptitle(f'Learned First-Layer Weights (showing {N_SHOW} of {W_first.shape[0]} neurons)', fontsize=12)
plt.tight_layout()
plt.show()

print('Observe: filters show digit-like patterns — strokes, curves, and edges.')
print('The model has learned to detect discriminative features automatically.')

## Part 8: Failure Analysis — Confusion Matrix and Misclassified Examples

Understanding failure modes is as important as achieving high accuracy.

The confusion matrix shows which pairs of digits are most often confused.
Classic confusions on MNIST: 4↔9, 3↔5, 7↔1 (visually similar strokes).

In [ ]:
# Collect all predictions on the test set
all_preds = []
all_labels = []
all_images = []
all_probs  = []

model_final.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images_d = images.to(DEVICE)
        logits = model_final(images_d)
        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        all_preds.append(preds.cpu())
        all_labels.append(labels)
        all_images.append(images)
        all_probs.append(probs.cpu())

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
all_images = torch.cat(all_images).numpy()   # (N, 1, 28, 28)
all_probs  = torch.cat(all_probs).numpy()    # (N, 10)

print(f'Test set: {len(all_labels):,} samples')
print(f'Correct: {(all_preds == all_labels).sum():,}  ({(all_preds == all_labels).mean():.4f})')

In [ ]:
# Build and plot confusion matrix
CONFUSION_MATRIX = np.zeros((N_CLASSES, N_CLASSES), dtype=int)
for pred, label in zip(all_preds, all_labels):
    CONFUSION_MATRIX[label, pred] += 1

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(CONFUSION_MATRIX, cmap='Blues')

# Annotate cells
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        val = CONFUSION_MATRIX[i, j]
        color = 'white' if val > CONFUSION_MATRIX.max() * 0.5 else 'black'
        ax.text(j, i, str(val), ha='center', va='center', fontsize=9, color=color)

ax.set_xticks(range(N_CLASSES))
ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(CLASS_NAMES)
ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title('Confusion Matrix on MNIST Test Set')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# Most confused pairs
off_diag = CONFUSION_MATRIX.copy()
np.fill_diagonal(off_diag, 0)
top_confusions_flat = np.argsort(off_diag.ravel())[::-1][:5]
print('\nMost confused pairs (true → predicted, count):')
for flat_idx in top_confusions_flat:
    true_label = flat_idx // N_CLASSES
    pred_label = flat_idx % N_CLASSES
    count = off_diag[true_label, pred_label]
    print(f'  {true_label} → {pred_label}: {count} errors')

In [ ]:
# Show misclassified examples with model confidence
misclassified_mask = all_preds != all_labels
wrong_images = all_images[misclassified_mask]     # (n_wrong, 1, 28, 28)
wrong_preds  = all_preds[misclassified_mask]
wrong_labels = all_labels[misclassified_mask]
wrong_probs  = all_probs[misclassified_mask]

N_SHOW_WRONG = 25
COLS_WRONG   = 5
ROWS_WRONG   = N_SHOW_WRONG // COLS_WRONG

fig, axes = plt.subplots(ROWS_WRONG, COLS_WRONG, figsize=(COLS_WRONG * 3, ROWS_WRONG * 3))

for i, ax in enumerate(axes.ravel()):
    if i >= N_SHOW_WRONG or i >= len(wrong_images):
        ax.set_visible(False)
        continue

    img = wrong_images[i, 0]    # (28, 28)
    pred = wrong_preds[i]
    true = wrong_labels[i]
    conf = wrong_probs[i, pred]

    ax.imshow(img, cmap='gray')
    ax.set_title(f'True: {true}\nPred: {pred} ({conf:.0%})', fontsize=9, color='red')
    ax.axis('off')

plt.suptitle(f'Misclassified Examples (showing {min(N_SHOW_WRONG, len(wrong_images))} of {len(wrong_images)} errors)', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Total misclassified: {misclassified_mask.sum()} / {len(all_labels)} = {misclassified_mask.mean():.2%} error rate')
print('Observe: many errors involve visually ambiguous digits (messy 4s that look like 9s, etc.)')

## Summary

We trained a PyTorch MLP on MNIST and ran several experiments:

| Experiment | Finding |
|---|---|
| SGD vs Adam | Adam converges faster in early epochs; both reach >97% |
| Dropout=0.0 vs 0.3 vs 0.5 | Higher dropout → smaller train/val gap |
| Weight visualization | First-layer neurons learn digit-like stroke patterns |
| Confusion matrix | Most errors on visually similar pairs (4/9, 3/5) |
| Final test accuracy | >97% ✓ |

### What's next?

- **`02_deep_learning/cnns/`** — convolutional networks use spatial structure to beat
  99% on MNIST with fewer parameters
- **`02_deep_learning/attention_transformers/`** — replace the FFN with self-attention
- **`04_mlops/`** — packaging, serving, and monitoring trained models